<a href="https://colab.research.google.com/github/Lochanpatil9/logistic-cost-optimisation/blob/main/scripts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import sqlite3

raw = pd.read_csv('/content/raw_shipments.csv')
def parse_mixed_date(s):
    for fmt in ('%Y-%m-%d', '%d/%m/%Y', '%d-%m-%Y'):
        try:
            return pd.to_datetime(s, format=fmt).strftime('%Y-%m-%d')
        except ValueError:
            continue
    return pd.NaT

raw['order_date'] = raw['order_date'].apply(parse_mixed_date)

raw['weight_kg'] = raw.groupby('transport_mode')['weight_kg'].transform(lambda x: x.fillna(x.median()))
raw['actual_cost_inr'] = raw.groupby('transport_mode')['actual_cost_inr'].transform(lambda x: x.fillna(x.median()))

conn = sqlite3.connect(':memory:')
raw.to_sql('raw_shipments', conn, index=False, if_exists='replace')

sql = open('/content/01_clean_shipments.sql').read()
conn.executescript(sql)

clean = pd.read_sql('SELECT * FROM shipments_clean', conn)
clean.to_csv('/content/shipments_clean.csv', index=False)

print(f"Raw rows: {len(raw)}")
print(f"Clean rows: {len(clean)}  (removed duplicates, {8481-len(raw)} true dupes already handled by DISTINCT, plus outliers)")
print(f"Rows dropped as outliers/incomplete: {len(raw) - len(clean)}")
print(clean.head())


Raw rows: 8481
Clean rows: 8301  (removed duplicates, 0 true dupes already handled by DISTINCT, plus outliers)
Rows dropped as outliers/incomplete: 180
  shipment_id  order_date warehouse_id warehouse_city store_id store_city  \
0   SHP103980  2024-05-01       WH-BPL         Bhopal   ST-034      Satna   
1   SHP103905  2024-07-27       WH-JBP       Jabalpur   ST-038      Dewas   
2   SHP101264  2024-08-16       WH-GWL        Gwalior   ST-037    Gwalior   
3   SHP100685  2024-08-22       WH-GWL        Gwalior   ST-029     Bhopal   
4   SHP102697  2024-12-14       WH-IND         Indore   ST-019   Jabalpur   

   distance_km  weight_kg transport_mode  planned_cost_inr  actual_cost_inr  \
0        293.4      371.5            LCV          10887.90         11408.13   
1        267.4      372.4            LCV          10896.52         11510.05   
2        296.7       60.9            LCV          10869.56         11518.53   
3        416.8      191.2   Medium Truck          26018.66         26

In [4]:
import pandas as pd

clean = pd.read_csv('/content/shipments_clean.csv')

route_capacity = {'Mini Truck': 750, 'LCV': 1500, 'Medium Truck': 3500}
route_rate = {'Mini Truck': 24, 'LCV': 37, 'Medium Truck': 58}

daily = (clean.groupby(['warehouse_id', 'store_id', 'store_city', 'order_date'])
         .agg(total_weight_kg=('weight_kg', 'sum'),
              distance_km=('distance_km', 'mean'),
              shipment_count=('shipment_id', 'count'),
              current_total_cost=('actual_cost_inr', 'sum'))
         .reset_index())

daily = daily[daily['shipment_count'] >= 2].reset_index(drop=True)

routes = (daily.groupby(['warehouse_id', 'store_id', 'store_city'])
          .agg(total_weight_kg=('total_weight_kg', 'sum'),
               distance_km=('distance_km', 'mean'),
               shipment_count=('shipment_count', 'sum'),
               trip_count=('order_date', 'count'),
               current_total_cost=('current_total_cost', 'sum'))
          .reset_index())

routes = routes[routes['trip_count'] >= 5].reset_index(drop=True)
daily.to_csv('/content/daily_trips.csv', index=False)
routes['distance_km'] = routes['distance_km'].round(1)
routes['total_weight_kg'] = routes['total_weight_kg'].round(0)
routes['current_total_cost'] = routes['current_total_cost'].round(0)

routes['medium_truck_trips_needed'] = (routes['total_weight_kg'] / route_capacity['Medium Truck']).apply(
    lambda x: max(1, round(x + 0.49)))
routes['lcv_trips_needed'] = (routes['total_weight_kg'] / route_capacity['LCV']).apply(
    lambda x: max(1, round(x + 0.49)))
routes['minitruck_trips_needed'] = (routes['total_weight_kg'] / route_capacity['Mini Truck']).apply(
    lambda x: max(1, round(x + 0.49)))

routes['medium_truck_cost'] = routes['medium_truck_trips_needed'] * routes['distance_km'] * route_rate['Medium Truck']
routes['lcv_cost'] = routes['lcv_trips_needed'] * routes['distance_km'] * route_rate['LCV']
routes['minitruck_cost'] = routes['minitruck_trips_needed'] * routes['distance_km'] * route_rate['Mini Truck']

routes.to_csv('/content/route_summary.csv', index=False)
print(routes.shape)
print(routes.head(10))
print("\nTotal current (fragmented) cost: ", routes['current_total_cost'].sum())


(61, 14)
  warehouse_id store_id  store_city  total_weight_kg  distance_km  \
0       WH-BPL   ST-005      Indore           6221.0        180.5   
1       WH-BPL   ST-007      Indore           9239.0        392.8   
2       WH-BPL   ST-008   Burhanpur           3213.0        319.0   
3       WH-BPL   ST-009     Vidisha           5247.0        318.1   
4       WH-BPL   ST-014      Bhopal           4025.0        261.8   
5       WH-BPL   ST-015    Jabalpur           3974.0        436.4   
6       WH-BPL   ST-016  Chhindwara           3732.0        178.0   
7       WH-BPL   ST-018       Satna           3826.0        233.5   
8       WH-BPL   ST-020      Ratlam           5103.0        279.0   
9       WH-BPL   ST-025    Mandsaur           4649.0        413.5   

   shipment_count  trip_count  current_total_cost  medium_truck_trips_needed  \
0              13           6            120942.0                          2   
1              20           9            357565.0                      

In [5]:
import pandas as pd
import pulp
import math

daily = pd.read_csv('/content/daily_trips.csv')
clean = pd.read_csv('/content/shipments_clean.csv')

capacity = {'Medium Truck': 3500, 'LCV': 1500, 'Mini Truck': 750}
rate = {'Medium Truck': 58, 'LCV': 37, 'Mini Truck': 24}
vehicle_types = list(capacity.keys())

mode_by_group = (clean.groupby(['warehouse_id', 'store_id', 'order_date'])['transport_mode']
                  .agg(lambda x: x.value_counts().idxmax())
                  .reset_index()
                  .rename(columns={'transport_mode': 'habitual_mode'}))
daily = daily.merge(mode_by_group, on=['warehouse_id', 'store_id', 'order_date'], how='left')

def trips_needed(weight, mode):
    return math.ceil(weight / capacity[mode])

daily['current_trips'] = daily.apply(lambda r: trips_needed(r['total_weight_kg'], r['habitual_mode']), axis=1)
daily['current_cost_recalc'] = daily['current_trips'] * daily['distance_km'] * daily['habitual_mode'].map(rate)
daily['week'] = pd.to_datetime(daily['order_date']).dt.isocalendar().week


import random
random.seed(7)

MINI_AVAILABLE_PROB = 0.35

final_rows = []
for (wh, week), grp in daily.groupby(['warehouse_id', 'week']):
    grp = grp.reset_index(drop=True)
    fleet_cap_mini = 1 if random.random() < MINI_AVAILABLE_PROB else 0
    FLEET_CAP = {'Medium Truck': 0, 'Mini Truck': fleet_cap_mini}

    prob = pulp.LpProblem("opt", pulp.LpMinimize)
    trips = {(i, v): pulp.LpVariable(f"t_{i}_{v}", lowBound=0, cat='Integer')
             for i in grp.index for v in vehicle_types}

    prob += pulp.lpSum(trips[i, v] * rate[v] * grp.loc[i, 'distance_km']
                        for i in grp.index for v in vehicle_types)
    for i in grp.index:
        prob += pulp.lpSum(trips[i, v] * capacity[v] for v in vehicle_types) >= grp.loc[i, 'total_weight_kg']
    for v in FLEET_CAP:
        prob += pulp.lpSum(trips[i, v] for i in grp.index) <= FLEET_CAP[v]

    prob.solve(pulp.PULP_CBC_CMD(msg=0))

    for i in grp.index:
        opt_cost = sum(trips[i, v].value() * rate[v] * grp.loc[i, 'distance_km'] for v in vehicle_types)
        grp.loc[i, 'opt_trips_medium'] = trips[i, 'Medium Truck'].value()
        grp.loc[i, 'opt_trips_lcv'] = trips[i, 'LCV'].value()
        grp.loc[i, 'opt_trips_mini'] = trips[i, 'Mini Truck'].value()
        grp.loc[i, 'opt_cost'] = opt_cost
    final_rows.append(grp)

result = pd.concat(final_rows, ignore_index=True)
result.to_csv('/content/optimization_result_daily.csv', index=False)

baseline = result['current_cost_recalc'].sum()
optimized = result['opt_cost'].sum()
days_observed = pd.to_datetime(daily['order_date']).dt.date.nunique()
scale = 365 / days_observed
print(f"Trip-groups modeled: {len(result)}")
print(f"Baseline cost: Rs {baseline:,.0f}  -> annualized Rs {baseline*scale:,.0f}")
print(f"Optimized cost: Rs {optimized:,.0f}  -> annualized Rs {optimized*scale:,.0f}")
print(f"Reduction: {(baseline-optimized)/baseline*100:.2f}%")
print(f"Annualized savings: Rs {(baseline-optimized)*scale:,.0f}  (~Rs {(baseline-optimized)*scale/100000:.1f} lakh)")


Trip-groups modeled: 659
Baseline cost: Rs 11,416,782  -> annualized Rs 16,027,405
Optimized cost: Rs 9,866,574  -> annualized Rs 13,851,152
Reduction: 13.58%
Annualized savings: Rs 2,176,254  (~Rs 21.8 lakh)


In [7]:
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

FONT = 'Arial'
BLUE = Font(name=FONT, color='0000FF')
BLACK = Font(name=FONT, color='000000')
BOLD = Font(name=FONT, bold=True)
HEADER_FILL = PatternFill('solid', start_color='1F4E78')
HEADER_FONT = Font(name=FONT, bold=True, color='FFFFFF')
YELLOW = PatternFill('solid', start_color='FFFF00')
GREEN_FILL = PatternFill('solid', start_color='C6EFCE')
THIN = Border(*[Side(style='thin', color='B7B7B7')]*4)

wb = Workbook()

# ============ SHEET 1: Solver Model (interactive, small illustrative week) ============
ws = wb.active
ws.title = 'Solver_Model'
ws.sheet_view.showGridLines = False

ws['A1'] = 'Route Vehicle-Mix Optimization — Solver Model'
ws['A1'].font = Font(name=FONT, bold=True, size=14)
ws['A2'] = 'Illustrative unit: Warehouse WH-JBP, Week 35 (8 delivery routes)'
ws['A2'].font = Font(name=FONT, italic=True, size=10)

ws['A4'] = 'Vehicle Assumptions'
ws['A4'].font = BOLD
headers = ['Vehicle Type', 'Capacity (kg)', 'Rate (Rs/km)']
for j, h in enumerate(headers):
    c = ws.cell(row=5, column=1+j, value=h)
    c.font = HEADER_FONT; c.fill = HEADER_FILL
vehicle_data = [('Medium Truck', 3500, 58), ('LCV', 1500, 37), ('Mini Truck', 750, 24)]
for i, (v, cap, rate) in enumerate(vehicle_data):
    r = 6+i
    ws.cell(row=r, column=1, value=v).font = BLACK
    ws.cell(row=r, column=2, value=cap).font = BLUE
    ws.cell(row=r, column=3, value=rate).font = BLUE
ws['A10'] = 'Fleet availability this week (owned Medium/Mini trucks are limited; LCV hired on-demand):'
ws['A10'].font = Font(name=FONT, italic=True, size=9)
ws['A11'] = 'Max Medium Truck trips available'
ws['C11'] = 0
ws['C11'].font = BLUE; ws['C11'].fill = YELLOW
ws['A12'] = 'Max Mini Truck trips available'
ws['C12'] = 1
ws['C12'].font = BLUE; ws['C12'].fill = YELLOW

# Route table
start_row = 15
ws.cell(row=start_row, column=1, value='Route Data & Decision Variables').font = BOLD
hdrs = ['Store', 'City', 'Weight (kg)', 'Distance (km)', 'Habitual Mode', 'Current Cost (Rs)',
        'Trips: Medium', 'Trips: LCV', 'Trips: Mini', 'Capacity Provided (kg)', 'Feasible?', 'Optimized Cost (Rs)']
for j, h in enumerate(hdrs):
    c = ws.cell(row=start_row+1, column=1+j, value=h)
    c.font = HEADER_FONT; c.fill = HEADER_FILL
    c.alignment = Alignment(wrap_text=True, horizontal='center')

demo = pd.read_csv('/content/solver_demo_week.csv')
# pre-populate decision vars with the solved optimum (from the LP solve) so the sheet
# opens already showing the answer; Solver can be re-run to reproduce it live.
solved_trips = {
    'ST-004': (0, 1, 0), 'ST-007': (0, 0, 1), 'ST-009': (0, 1, 0), 'ST-012': (0, 1, 0),
    'ST-016': (0, 1, 0), 'ST-026': (0, 1, 0), 'ST-031': (0, 1, 0), 'ST-038': (0, 1, 0),
}

data_start = start_row + 2
for i, row in demo.iterrows():
    r = data_start + i
    ws.cell(row=r, column=1, value=row['store_id']).font = BLACK
    ws.cell(row=r, column=2, value=row['store_city']).font = BLACK
    ws.cell(row=r, column=3, value=row['total_weight_kg']).font = BLUE
    ws.cell(row=r, column=4, value=row['distance_km']).font = BLUE
    ws.cell(row=r, column=5, value=row['habitual_mode']).font = BLACK
    ws.cell(row=r, column=6, value=row['current_cost_recalc']).font = BLACK
    med, lcv, mini = solved_trips[row['store_id']]
    ws.cell(row=r, column=7, value=med).font = BLUE
    ws.cell(row=r, column=8, value=lcv).font = BLUE
    ws.cell(row=r, column=9, value=mini).font = BLUE
    ws.cell(row=r, column=10, value=f'=G{r}*$B$6+H{r}*$B$7+I{r}*$B$8').font = BLACK
    ws.cell(row=r, column=11, value=f'=IF(J{r}>=C{r},"OK","SHORT")').font = BLACK
    ws.cell(row=r, column=12, value=f'=(G{r}*$C$6+H{r}*$C$7+I{r}*$C$8)*D{r}').font = BLACK

last_data_row = data_start + len(demo) - 1
total_row = last_data_row + 2
ws.cell(row=total_row, column=5, value='TOTAL').font = BOLD
ws.cell(row=total_row, column=6, value=f'=SUM(F{data_start}:F{last_data_row})').font = BOLD
ws.cell(row=total_row, column=12, value=f'=SUM(L{data_start}:L{last_data_row})').font = BOLD
ws.cell(row=total_row, column=12).fill = GREEN_FILL

ws.cell(row=total_row+1, column=5, value='Reduction %').font = BOLD
ws.cell(row=total_row+1, column=6, value=f'=(F{total_row}-L{total_row})/F{total_row}').font = BOLD
ws.cell(row=total_row+1, column=6).number_format = '0.0%'
ws.cell(row=total_row+1, column=6).fill = GREEN_FILL

ws.cell(row=total_row+3, column=1, value='Fleet constraint check:').font = Font(name=FONT, italic=True)
ws.cell(row=total_row+4, column=1, value='Total Medium trips used')
ws.cell(row=total_row+4, column=3, value=f'=SUM(G{data_start}:G{last_data_row})').font = BLACK
ws.cell(row=total_row+4, column=4, value='<= C11 limit')
ws.cell(row=total_row+5, column=1, value='Total Mini trips used')
ws.cell(row=total_row+5, column=3, value=f'=SUM(I{data_start}:I{last_data_row})').font = BLACK
ws.cell(row=total_row+5, column=4, value='<= C12 limit')

# Objective cell reference
ws.cell(row=total_row+7, column=1, value='OBJECTIVE CELL (minimize):').font = BOLD
ws.cell(row=total_row+7, column=3, value=f'L{total_row}').font = Font(name=FONT, bold=True, color='FF0000')

# How to run Solver
instr_row = total_row + 10
ws.cell(row=instr_row, column=1, value='How to reproduce this in Excel Solver:').font = BOLD
steps = [
    '1. Data tab -> Solver (install via File > Options > Add-ins > Solver Add-in if not visible)',
    f'2. Set Objective: L{total_row}  |  To: Min',
    f'3. By Changing Variable Cells: G{data_start}:I{last_data_row}',
    f'4. Add Constraint: J{data_start}:J{last_data_row} >= C{data_start}:C{last_data_row}  (capacity meets demand)',
    f'5. Add Constraint: SUM(G{data_start}:G{last_data_row}) <= $C$11  (Medium Truck fleet cap)',
    f'6. Add Constraint: SUM(I{data_start}:I{last_data_row}) <= $C$12  (Mini Truck fleet cap)',
    f'7. Add Constraint: G{data_start}:I{last_data_row} = integer',
    '8. Solving Method: Simplex LP  ->  Solve',
]
for i, s in enumerate(steps):
    ws.cell(row=instr_row+1+i, column=1, value=s).font = Font(name=FONT, size=9)

for col, w in zip('ABCDEFGHIJKL', [10,12,11,12,14,15,10,9,10,17,10,15]):
    ws.column_dimensions[col].width = w

# ============ SHEET 2: Results Summary (full network, all 157 routes) ============
ws2 = wb.create_sheet('Results_Summary')
ws2.sheet_view.showGridLines = False

ws2['A1'] = 'Logistics Cost Optimization — Results Summary'
ws2['A1'].font = Font(name=FONT, bold=True, size=14)
ws2['A2'] = f'Full network: 4 warehouses, 40 stores, {8301} cleaned shipment records, Apr–Dec 2024'
ws2['A2'].font = Font(name=FONT, italic=True, size=10)

routes = pd.read_csv('/content/route_final_summary.csv')

kpi_row = 4
table_start = kpi_row + 8

ws2.cell(row=kpi_row, column=1, value='KEY RESULTS').font = BOLD
kpis = [
    ('Total observed logistics spend (9-month sample)', f"=SUM(F{table_start}:F{table_start+len(routes)-1})", '#,##0'),
    ('Optimized spend (Solver-recommended vehicle mix)', f"=SUM(G{table_start}:G{table_start+len(routes)-1})", '#,##0'),
    ('Cost reduction', None, '0.0%'),
    ('Annualized savings (Rs, scaled to 365 days)', None, '#,##0'),
]
r = kpi_row + 1
ws2.cell(row=r, column=1, value=kpis[0][0]).font = BLACK
ws2.cell(row=r, column=3, value=kpis[0][1]).font = Font(name=FONT, bold=True)
ws2.cell(row=r, column=3).number_format = kpis[0][2]
r += 1
ws2.cell(row=r, column=1, value=kpis[1][0]).font = BLACK
ws2.cell(row=r, column=3, value=kpis[1][1]).font = Font(name=FONT, bold=True)
ws2.cell(row=r, column=3).number_format = kpis[1][2]
r += 1
ws2.cell(row=r, column=1, value=kpis[2][0]).font = BLACK
ws2.cell(row=r, column=3, value=f'=(C{kpi_row+1}-C{kpi_row+2})/C{kpi_row+1}').font = Font(name=FONT, bold=True, color='008000')
ws2.cell(row=r, column=3).number_format = kpis[2][2]
r += 1
ws2.cell(row=r, column=1, value=kpis[3][0]).font = BLACK
days_observed = 260
ws2.cell(row=r, column=3, value=f'=(C{kpi_row+1}-C{kpi_row+2})*365/{days_observed}').font = Font(name=FONT, bold=True, color='008000')
ws2.cell(row=r, column=3).number_format = kpis[3][2]

hdrs2 = ['Warehouse', 'Store', 'City', 'Trips', 'Total Weight (kg)', 'Current Cost (Rs)',
         'Optimized Cost (Rs)', 'Savings (Rs)', 'Savings %']
for j, h in enumerate(hdrs2):
    c = ws2.cell(row=table_start-1, column=1+j, value=h)
    c.font = HEADER_FONT; c.fill = HEADER_FILL
    c.alignment = Alignment(wrap_text=True, horizontal='center')

for i, row in routes.iterrows():
    r = table_start + i
    ws2.cell(row=r, column=1, value=row['warehouse_id']).font = BLACK
    ws2.cell(row=r, column=2, value=row['store_id']).font = BLACK
    ws2.cell(row=r, column=3, value=row['store_city']).font = BLACK
    ws2.cell(row=r, column=4, value=row['trip_count']).font = BLACK
    ws2.cell(row=r, column=5, value=row['total_weight_kg']).font = BLACK
    ws2.cell(row=r, column=6, value=row['current_cost']).font = BLACK
    ws2.cell(row=r, column=7, value=row['optimized_cost']).font = BLACK
    ws2.cell(row=r, column=8, value=f'=F{r}-G{r}').font = BLACK
    ws2.cell(row=r, column=9, value=f'=H{r}/F{r}').font = BLACK
    ws2.cell(row=r, column=9).number_format = '0.0%'

for col, w in zip('ABCDEFGHI', [12, 9, 12, 8, 14, 15, 16, 13, 10]):
    ws2.column_dimensions[col].width = w

import os
os.makedirs('/content/logistics-project/excel', exist_ok=True)
wb.save('/content/Logistics_Route_Optimization_Model.xlsx')
print('saved')


saved
